In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Rutas
DATA_RAW = Path("../data/raw")

# Para que los dataframes se vean completos en el notebook
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

print("Librerías cargadas correctamente")

Matplotlib is building the font cache; this may take a moment.


Librerías cargadas correctamente


In [6]:
# Carga de los tres CSVs
books = pd.read_csv(DATA_RAW / "books.csv", encoding="utf-8", low_memory=False, on_bad_lines="skip")
copies = pd.read_csv(DATA_RAW / "copies(ejemplares).csv", encoding="utf-8")
users = pd.read_csv(DATA_RAW / "user_info.csv", encoding="utf-8")
ratings = pd.read_csv(DATA_RAW / "ratings.csv", encoding="utf-8")

# Resumen inicial
for nombre, df in [("books", books), ("copies", copies), ("users", users), ("ratings", ratings)]:
    print(f"{'='*40}")
    print(f"{nombre.upper()}")
    print(f"  Filas: {df.shape[0]:,}  |  Columnas: {df.shape[1]}")
    print(f"  Columnas: {list(df.columns)}")
print("="*40)

BOOKS
  Filas: 9,998  |  Columnas: 8
  Columnas: ['isbn', 'authors', 'original_publication_year', 'original_title', 'title', 'language_code', 'book_id', 'image_url']
COPIES
  Filas: 55,327  |  Columnas: 2
  Columnas: ['copy_id', 'book_id']
USERS
  Filas: 501  |  Columnas: 4
  Columnas: ['user_id', 'sexo', 'comentario', 'fecha_nacimiento']
RATINGS
  Filas: 5,976,479  |  Columnas: 3
  Columnas: ['user_id', 'copy_id', 'rating']


In [7]:
# ── BOOKS: Nulos por columna ──────────────────────────────
print("BOOKS — Nulos por columna")
print("="*40)
nulos_books = books.isnull().sum()
pct_books = (nulos_books / len(books) * 100).round(2)
resumen_nulos_books = pd.DataFrame({"nulos": nulos_books, "pct": pct_books})
print(resumen_nulos_books)

BOOKS — Nulos por columna
                           nulos    pct
isbn                         700   7.00
authors                        0   0.00
original_publication_year     21   0.21
original_title               585   5.85
title                          0   0.00
language_code               1084  10.84
book_id                        0   0.00
image_url                      0   0.00


In [8]:
# ── BOOKS: Duplicados ─────────────────────────────────────
print("BOOKS — Duplicados")
print("="*40)
dup_totales = books.duplicated().sum()
dup_book_id = books.duplicated(subset="book_id").sum()
dup_isbn = books.duplicated(subset="isbn").sum()

print(f"  Filas completamente duplicadas: {dup_totales}")
print(f"  book_id duplicados (PK):        {dup_book_id}")
print(f"  isbn duplicados:                {dup_isbn}")

BOOKS — Duplicados
  Filas completamente duplicadas: 0
  book_id duplicados (PK):        0
  isbn duplicados:                699


In [9]:
# ── BOOKS: Verificación ISBN duplicados ───────────────────
print("BOOKS — Verificación ISBN duplicados")
print("="*40)

# ISBNs duplicados excluyendo nulos
dup_isbn_real = books.duplicated(subset="isbn", keep=False) & books["isbn"].notna()
print(f"  ISBN duplicados reales (sin contar nulos): {dup_isbn_real.sum()}")

# Muestra los casos reales si los hay
if dup_isbn_real.sum() > 0:
    print(books[dup_isbn_real][["book_id", "isbn", "title"]].sort_values("isbn"))
else:
    print("  → No hay ISBNs duplicados reales. Los 699 son los NaN contados como iguales.")

BOOKS — Verificación ISBN duplicados
  ISBN duplicados reales (sin contar nulos): 0
  → No hay ISBNs duplicados reales. Los 699 son los NaN contados como iguales.


In [10]:
# ── BOOKS: Formatos y valores sospechosos ─────────────────
print("BOOKS — Formatos y valores sospechosos")
print("="*40)

# Años fuera de rango
year_min, year_max = 1450, 2025
years = books["original_publication_year"].dropna()
fuera_rango = books[
    (books["original_publication_year"] < year_min) |
    (books["original_publication_year"] > year_max)
]
print(f"  Años fuera de rango ({year_min}-{year_max}): {len(fuera_rango)}")
if len(fuera_rango) > 0:
    print(fuera_rango[["book_id", "title", "original_publication_year"]])

# Distribución de language_code
print(f"\n  Distribución language_code (top 10):")
print(books["language_code"].value_counts(dropna=False).head(10))

# ISBNs con formato raro (distinto de 10 o 13 dígitos)
isbn_validos = books["isbn"].dropna().astype(str).str.strip()
isbn_formato_raro = isbn_validos[~isbn_validos.str.match(r"^\d{10}$|^\d{13}$|^\d{9}X$")]
print(f"\n  ISBNs con formato inesperado: {len(isbn_formato_raro)}")
if len(isbn_formato_raro) > 0:
    print(isbn_formato_raro.head(10))

BOOKS — Formatos y valores sospechosos
  Años fuera de rango (1450-2025): 49
      book_id  \
85       9304   
169        79   
430       341   
492       403   
620       531   
732       646   
858       772   
910       824   
1064      977   
1184     1099   
1206     1120   
1252     1167   
1363     1280   
1605     1521   
1990     1910   
2046     1967   
2131     2052   
2155     2076   
2161     2082   
2220     2142   
2414     2336   
2444     2366   
2843     2763   
3099     3022   
3248     3172   
3332     3256   
3479     3404   
3937     3868   
3999     3930   
4016     3949   
4211     4149   
4392     4330   
4394     4332   
4497     4435   
4598     4537   
4936     4880   
5683     5637   
5989     5940   
6212     6166   
6563     6524   
6806     6772   
6928     6896   
7796     7778   
8483     8472   
8643     8633   
8953     8946   
8956     8949   
9285     9281   
9679     9679   

                                                                        

In [11]:
# ── BOOKS: Impacto de los criterios de descarte del cliente ──
print("BOOKS — Registros que se perderían con los criterios del cliente")
print("="*40)

sin_isbn    = books["isbn"].isnull()
sin_authors = books["authors"].isnull()
sin_year    = books["original_publication_year"].isnull()

descarte = sin_isbn | sin_authors | sin_year
total_descarte = descarte.sum()

print(f"  Sin ISBN:   {sin_isbn.sum()}")
print(f"  Sin autor:  {sin_authors.sum()}")
print(f"  Sin año:    {sin_year.sum()}")
print(f"  ─────────────────────────────")
print(f"  Total a descartar (unión): {total_descarte} de {len(books)} ({total_descarte/len(books)*100:.1f}%)")
print(f"  Quedarían en la BD:        {len(books) - total_descarte}")

# Guardamos las filas descartadas para el informe
books[descarte].to_csv("../notebooks/outputs/books_descartes.csv", index=False)
print(f"\n  → Exportado a notebooks/outputs/books_descartes.csv")

BOOKS — Registros que se perderían con los criterios del cliente
  Sin ISBN:   700
  Sin autor:  0
  Sin año:    21
  ─────────────────────────────
  Total a descartar (unión): 718 de 9998 (7.2%)
  Quedarían en la BD:        9280

  → Exportado a notebooks/outputs/books_descartes.csv


In [12]:
# ── USERS: Nulos y duplicados ─────────────────────────────
print("USERS — Nulos por columna")
print("="*40)
nulos_users = users.isnull().sum()
pct_users = (nulos_users / len(users) * 100).round(2)
print(pd.DataFrame({"nulos": nulos_users, "pct": pct_users}))

print(f"\nDuplicados")
print("="*40)
print(f"  Filas completamente duplicadas: {users.duplicated().sum()}")
print(f"  user_id duplicados (PK):        {users.duplicated(subset='user_id').sum()}")

USERS — Nulos por columna
                  nulos  pct
user_id               0  0.0
sexo                  0  0.0
comentario            0  0.0
fecha_nacimiento      0  0.0

Duplicados
  Filas completamente duplicadas: 0
  user_id duplicados (PK):        0


In [13]:
# ── USERS: Fechas de nacimiento y edades ──────────────────
print("USERS — Análisis fecha_nacimiento")
print("="*40)

# Convertir a datetime
users["fecha_nacimiento_dt"] = pd.to_datetime(users["fecha_nacimiento"], format="%d/%m/%Y", errors="coerce")

# Cuántas no se pudieron parsear
errores_fecha = users["fecha_nacimiento_dt"].isnull().sum()
print(f"  Fechas con formato incorrecto: {errores_fecha}")

# Calcular edades
hoy = pd.Timestamp("2026-05-10")
users["edad"] = ((hoy - users["fecha_nacimiento_dt"]).dt.days / 365.25).round(1)

print(f"\n  Edad mínima:  {users['edad'].min()}")
print(f"  Edad máxima:  {users['edad'].max()}")
print(f"  Edad media:   {users['edad'].mean():.1f}")

# Edades sospechosas (menores de 6 o mayores de 100)
sospechosas = users[(users["edad"] < 6) | (users["edad"] > 100)]
print(f"\n  Edades fuera de rango (<6 o >100): {len(sospechosas)}")
if len(sospechosas) > 0:
    print(sospechosas[["user_id", "fecha_nacimiento", "edad"]])

USERS — Análisis fecha_nacimiento
  Fechas con formato incorrecto: 0

  Edad mínima:  20.4
  Edad máxima:  99.7
  Edad media:   33.9

  Edades fuera de rango (<6 o >100): 0


In [14]:
# ── COPIES: Nulos y duplicados ────────────────────────────
print("COPIES — Nulos por columna")
print("="*40)
nulos_copies = copies.isnull().sum()
pct_copies = (nulos_copies / len(copies) * 100).round(2)
print(pd.DataFrame({"nulos": nulos_copies, "pct": pct_copies}))

print(f"\nDuplicados")
print("="*40)
print(f"  Filas completamente duplicadas: {copies.duplicated().sum()}")
print(f"  copy_id duplicados (PK):        {copies.duplicated(subset='copy_id').sum()}")

COPIES — Nulos por columna
         nulos  pct
copy_id      0  0.0
book_id      0  0.0

Duplicados
  Filas completamente duplicadas: 0
  copy_id duplicados (PK):        0


In [16]:
# ── INTEGRIDAD REFERENCIAL ────────────────────────────────
print("INTEGRIDAD REFERENCIAL")
print("="*40)

# 1. book_id en copies que no existe en books
book_ids_books = set(books["book_id"])
book_ids_copies = set(copies["book_id"])
huerfanos_copies = book_ids_copies - book_ids_books
print(f"  book_id en copies sin libro en books:    {len(huerfanos_copies)}")

# 2. copy_id en ratings que no existe en copies
copy_ids_copies = set(copies["copy_id"])
copy_ids_ratings = set(ratings["copy_id"])
huerfanos_ratings = copy_ids_ratings - copy_ids_copies
print(f"  copy_id en ratings sin copia en copies:  {len(huerfanos_ratings)}")

# 3. user_id en ratings que no existe en users
user_ids_users = set(users["user_id"])
user_ids_ratings = set(ratings["user_id"])
huerfanos_users = user_ids_ratings - user_ids_users
print(f"  user_id en ratings sin usuario en users: {len(huerfanos_users)}")

# 4. Cuántos ratings se perderían si aplicamos los descartes de books
books_validos = books[books["isbn"].notna() & books["original_publication_year"].notna()]["book_id"]
copies_validas = copies[copies["book_id"].isin(books_validos)]["copy_id"]
ratings_validos = ratings[ratings["copy_id"].isin(copies_validas)]
ratings_perdidos = len(ratings) - len(ratings_validos)
print(f"\n  Ratings perdidos al descartar libros sin ISBN/año:")
print(f"  {ratings_perdidos:,} de {len(ratings):,} ({ratings_perdidos/len(ratings)*100:.1f}%)")

# Exportar huerfanos para el informe
pd.DataFrame({
    "tipo": (["book_id huerfano en copies"] * len(huerfanos_copies) +
             ["copy_id huerfano en ratings"] * len(huerfanos_ratings) +
             ["user_id huerfano en ratings"] * len(huerfanos_users)),
    "id": list(huerfanos_copies) + list(huerfanos_ratings) + list(huerfanos_users)
}).to_csv("../notebooks/outputs/integridad_referencial.csv", index=False)
print("\n  -> Exportado a notebooks/outputs/integridad_referencial.csv")

INTEGRIDAD REFERENCIAL
  book_id en copies sin libro en books:    2
  copy_id en ratings sin copia en copies:  0
  user_id en ratings sin usuario en users: 52924

  Ratings perdidos al descartar libros sin ISBN/año:
  218,535 de 5,976,479 (3.7%)

  -> Exportado a notebooks/outputs/integridad_referencial.csv


In [17]:
# ── RATINGS: Nulos, duplicados y rango de valores ─────────
print("RATINGS — Nulos por columna")
print("="*40)
nulos_ratings = ratings.isnull().sum()
pct_ratings = (nulos_ratings / len(ratings) * 100).round(2)
print(pd.DataFrame({"nulos": nulos_ratings, "pct": pct_ratings}))

print(f"\nDuplicados")
print("="*40)
print(f"  Filas completamente duplicadas:          {ratings.duplicated().sum():,}")
print(f"  PKs duplicadas (user_id + copy_id):      {ratings.duplicated(subset=['user_id','copy_id']).sum():,}")

print(f"\nRango de valores rating")
print("="*40)
print(ratings["rating"].value_counts().sort_index())
fuera_rango = ratings[(ratings["rating"] < 1) | (ratings["rating"] > 5)]
print(f"\n  Ratings fuera de rango (1-5): {len(fuera_rango)}")

RATINGS — Nulos por columna
         nulos  pct
user_id      0  0.0
copy_id      0  0.0
rating       0  0.0

Duplicados
  Filas completamente duplicadas:          0
  PKs duplicadas (user_id + copy_id):      0

Rango de valores rating
rating
1     124195
2     359257
3    1370916
4    2139018
5    1983093
Name: count, dtype: int64

  Ratings fuera de rango (1-5): 0


In [19]:
# ── RESUMEN EJECUTIVO ─────────────────────────────────────
lineas = [
    "# Auditoria de calidad de datos - Casa de la Cultura",
    "Fecha: 2026-05-10",
    "",
    "## Volumen de datos",
    "| Tabla   | Filas      |",
    "|---------|------------|",
    "| books   | 9.998      |",
    "| copies  | 55.327     |",
    "| users   | 501        |",
    "| ratings | 5.976.479  |",
    "",
    "## Hallazgos por tabla",
    "",
    "### books",
    "- 2 filas rechazadas al cargar (campos mal delimitados)",
    "- ISBN nulos: 700 (7%) — criterio de descarte del cliente",
    "- Año de publicacion nulo: 21 (0.21%) — criterio de descarte del cliente",
    "- language_code nulo: 1.084 (10.84%) — requiere decision en el ETL",
    "- original_title nulo: 585 (5.85%) — aceptable",
    "- Sin duplicados reales ni en PK ni en ISBN",
    "- 49 libros con año negativo (fechas a.C.) — clasicos antiguos, datos correctos",
    "- 6.600 ISBNs de 9 digitos — ISBN-10 sin cero inicial, recuperables con zero-padding",
    "- language_code fragmentado: eng/en-US/en-GB/en-CA — normalizar en el ETL",
    "- **Registros a descartar: 718 (7.2%) -> quedan 9.280 libros**",
    "",
    "### copies",
    "- Sin nulos ni duplicados",
    "- 2 book_id sin correspondencia en books (coincide con filas rechazadas al cargar)",
    "",
    "### users",
    "- Sin nulos ni duplicados",
    "- Fechas todas en formato dd/mm/yyyy correcto",
    "- Edades entre 20 y 99 anos, sin valores absurdos",
    "",
    "### ratings",
    "- Sin nulos ni duplicados",
    "- PKs compuestas (user_id + copy_id) unicas",
    "- Valores entre 1 y 5, sin outliers",
    "- Distribucion con sesgo positivo (tipico en recomendacion)",
    "- **52.924 user_id en ratings sin registro en users — CSV de usuarios incompleto**",
    "",
    "## Integridad referencial",
    "| Relacion                  | Huerfanos |",
    "|---------------------------|-----------|",
    "| copies -> books           | 2         |",
    "| ratings -> copies         | 0         |",
    "| ratings -> users          | 52.924    |",
    "",
    "## Impacto de los descartes en ratings",
    "Al eliminar los 718 libros sin ISBN o año, se pierden 218.535 ratings (3.7%).",
    "",
    "## Decisiones pendientes para el ETL",
    "1. ISBNs de 9 digitos: aplicar zero-padding para recuperarlos",
    "2. language_code: normalizar variantes de ingles a 'eng'",
    "3. Ratings con user_id sin usuario: decidir si se conservan o descartan",
    "4. Anos negativos (a.C.): conservar tal cual, son datos correctos",
]

with open("../notebooks/outputs/informe_auditoria.md", "w", encoding="utf-8") as f:
    f.write("\n".join(lineas))

print("Informe exportado a notebooks/outputs/informe_auditoria.md")
print("Auditoria completada.")

Informe exportado a notebooks/outputs/informe_auditoria.md
Auditoria completada.
